# LSTM Storm Path Visualization
This notebook visualizes the actual vs LSTM-predicted storm paths for 6h, 12h, 18h, and 24h horizons.

In [ ]:
import os
import sys
import torch
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Add current dir to path to import local modules
sys.path.append(os.path.abspath('.'))
from dataset import preprocess_data, create_sequences
from model import StormLSTM

In [ ]:
# Load paths
TEST_DATA_PATH = r'..\..\..\Datasets\Storm_data\test.csv'
MODEL_PATH = r'models\lstm_model.pth'
SCALER_PATH = r'models\scaler.joblib'

# 1. Load Data
df_test = pd.read_csv(TEST_DATA_PATH, low_memory=False)
df_test, feature_cols = preprocess_data(df_test)

# Pick a specific storm
storm_id = df_test['international_id'].unique()[0]
storm_df = df_test[df_test['international_id'] == storm_id].copy()
print(f"Visualizing Storm ID: {storm_id}")

In [ ]:
# 2. Extract sequences for this single storm
seq_len = 4
horizons = 4
X, Y, base_lat_lon = create_sequences(storm_df, feature_cols, seq_len=seq_len, horizons=horizons)

# 3. Scale input
scaler = joblib.load(SCALER_PATH)
num_features = len(feature_cols)
X_flat = X.reshape(-1, num_features)
X_scaled = scaler.transform(X_flat).reshape(X.shape)

# 4. Load Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = StormLSTM(input_dim=num_features, hidden_dim=64, num_layers=2, output_dim=horizons*2).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

# 5. Predict
with torch.no_grad():
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)
    preds = model(X_tensor).cpu().numpy()

In [ ]:
# 6. Reconstruct Path (we will visualize the predictions made at a specific time step)
# Change state_idx to predict from any state of the storm (from 0 to len(preds)-1)
state_idx = len(preds) // 2  # Predict from the middle of the storm's life

pred_deltas = preds[state_idx]
true_deltas = Y[state_idx]
base_pos = base_lat_lon[state_idx]

pred_lat = [base_pos[0]] + [base_pos[0] + pred_deltas[i*2] for i in range(horizons)]
pred_lon = [base_pos[1]] + [base_pos[1] + pred_deltas[i*2+1] for i in range(horizons)]

true_lat = [base_pos[0]] + [base_pos[0] + true_deltas[i*2] for i in range(horizons)]
true_lon = [base_pos[1]] + [base_pos[1] + true_deltas[i*2+1] for i in range(horizons)]

# Also get the past trajectory (the input sequence)
past_lat = X[state_idx, :, feature_cols.index('lat')]
past_lon = X[state_idx, :, feature_cols.index('lon')]

In [ ]:
# 7. Plotting
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')

ax.plot(past_lon, past_lat, marker='o', color='black', label='Past 24h Trajectory', transform=ccrs.PlateCarree())
ax.plot(true_lon, true_lat, marker='s', color='green', linestyle='-', label='True Future Path (24h)', transform=ccrs.PlateCarree())
ax.plot(pred_lon, pred_lat, marker='^', color='red', linestyle='--', label='LSTM Predicted Path (24h)', transform=ccrs.PlateCarree())
ax.plot(base_pos[1], base_pos[0], marker='*', color='yellow', markersize=15, markeredgecolor='black', label='Current Position', transform=ccrs.PlateCarree())

# Zoom map to the storm area
all_lons = list(past_lon) + true_lon + pred_lon
all_lats = list(past_lat) + true_lat + pred_lat
margin = 5
ax.set_extent([min(all_lons)-margin, max(all_lons)+margin, min(all_lats)-margin, max(all_lats)+margin], crs=ccrs.PlateCarree())

plt.title(f'LSTM Storm Path Prediction: Storm {storm_id} (State {state_idx})')
plt.legend()
plt.grid(True)
plt.show()